<a href="https://colab.research.google.com/github/tousifo/ml_notebooks/blob/main/Bangladesh_Multi_Hazard_AI_Simulator_All_In_One_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bangladesh Multi-Hazard AI Simulator — All-in-One Colab Notebook

This notebook builds a full Bangladesh multi-hazard risk simulator and writes a polished Streamlit app from inside Colab.

**Core idea:** the app predicts transparent 0–10 risk indices for flood, cyclone, drought, agricultural stress and composite vulnerability. It does **not** claim exact disaster-event forecasting.

## Why the old R² looked high

The earlier Streamlit code manually replaced weak model scores with positive values. This notebook removes that unsafe shortcut and uses transparent engineered risk indices instead. The model leaderboard reports real test R² values.

In [3]:
# ============================================================
# 1) Install and Import Dependencies
# ============================================================

!pip -q install streamlit plotly scikit-learn pandas numpy

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

print("✅ Environment ready.")

✅ Environment ready.


In [4]:
# ============================================================
# 2) Dataset Check
# ============================================================

DATA_PATH = "/content/Bangladesh_Environmental_Climate_Change_Impact.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

required_columns = [
    "District",
    "Year",
    "Avg_Temperature_C",
    "Annual_Rainfall_mm",
    "AQI",
    "Forest_Cover_Percent",
    "Flood_Impact_Score",
    "Drought_Severity",
    "River_Water_Level_m",
    "Cyclone_Count",
    "Carbon_Emission_Metric_Tons_per_Capita",
    "Agricultural_Yield_ton_per_hectare"
]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    raise ValueError(f"❌ Missing required columns: {missing}")

print("✅ Required columns found.")
print("Latest dataset year:", df["Year"].max())
print("Total records:", len(df))
print("Total districts:", df["District"].nunique())

Dataset shape: (3000, 15)


,Year,District,Avg_Temperature_C,Annual_Rainfall_mm,AQI,Forest_Cover_Percent,River_Water_Level_m,Cyclone_Count,Flood_Impact_Score,Drought_Severity,Agricultural_Yield_ton_per_hectare,Coastal_Erosion_m_per_year,Urbanization_Rate_Percent,Carbon_Emission_Metric_Tons_per_Capita,Renewable_Energy_Usage_Percent
0,2025,Satkhira,27.06,4044.6,257,9.36,5.89,1,6.01,9.25,5.21,1.15,54.15,1.20,18.20
1,2018,Tangail,25.86,1659.1,222,29.12,8.74,1,9.55,6.24,4.28,2.09,47.95,2.02,16.47
2,2013,Sunamganj,25.20,3618.1,56,24.01,8.45,1,7.47,4.83,2.90,4.41,65.13,1.77,11.23
3,2005,Sirajganj,27.85,4031.3,171,25.68,4.04,2,1.80,5.43,5.20,1.10,32.84,0.69,14.80
4,2013,Barishal,29.94,3844.4,259,8.13,9.83,3,0.10,9.89,4.65,4.17,65.95,1.33,19.98


✅ Required columns found.
Latest dataset year: 2025
Total records: 3000
Total districts: 54


## Risk-index logic

The raw dataset columns such as `Flood_Impact_Score`, `Cyclone_Count`, and `Drought_Severity` may be noisy. To make the simulator defensible, we create transparent risk indices from rainfall, river level, temperature, AQI, forest cover, emissions and district exposure flags.

In [5]:
# ============================================================
# 3) Multi-Hazard Risk Index Engineering
# ============================================================

df_model = df.copy()

flood_districts = [
    "Sunamganj", "Sylhet", "Kurigram", "Gaibandha",
    "Jamalpur", "Bogura", "Sirajganj", "Lalmonirhat",
    "Netrokona", "Sherpur"
]

cyclone_districts = [
    "Bhola", "Patuakhali", "Satkhira", "Cox's Bazar",
    "Barguna", "Chittagong", "Chattogram", "Khulna",
    "Barisal", "Feni", "Noakhali", "Bagerhat"
]

drought_districts = [
    "Rajshahi", "Naogaon", "Chapainawabganj",
    "Dinajpur", "Rangpur", "Kushtia",
    "Thakurgaon", "Meherpur", "Chuadanga"
]

coords = {
    "Bagerhat": [22.6602, 89.7895],
    "Bandarban": [22.1953, 92.2184],
    "Barguna": [22.1591, 90.1189],
    "Barisal": [22.7010, 90.3535],
    "Bhola": [22.6859, 90.6440],
    "Bogura": [24.8481, 89.3730],
    "Brahmanbaria": [23.9578, 91.1111],
    "Chandpur": [23.2333, 90.6333],
    "Chapainawabganj": [24.5965, 88.2775],
    "Chattogram": [22.3569, 91.7832],
    "Chittagong": [22.3569, 91.7832],
    "Chuadanga": [23.6402, 88.8418],
    "Comilla": [23.4682, 91.1788],
    "Cox's Bazar": [21.4272, 92.0058],
    "Dhaka": [23.8103, 90.4125],
    "Dinajpur": [25.6217, 88.6354],
    "Feni": [23.0159, 91.3976],
    "Gaibandha": [25.3288, 89.5280],
    "Gazipur": [24.0023, 90.4260],
    "Gopalganj": [23.0051, 89.8266],
    "Jamalpur": [24.9375, 89.9375],
    "Jashore": [23.1664, 89.2081],
    "Jessore": [23.1664, 89.2081],
    "Khulna": [22.8456, 89.5403],
    "Kurigram": [25.8103, 89.6487],
    "Kushtia": [23.9010, 89.1204],
    "Lalmonirhat": [25.9167, 89.4500],
    "Magura": [23.4855, 89.4198],
    "Manikganj": [23.8617, 90.0003],
    "Maulvibazar": [24.4829, 91.7774],
    "Meherpur": [23.7622, 88.6318],
    "Mymensingh": [24.7471, 90.4203],
    "Naogaon": [24.7936, 88.9318],
    "Narayanganj": [23.6238, 90.5000],
    "Narsingdi": [23.9322, 90.7154],
    "Netrokona": [24.8835, 90.7310],
    "Noakhali": [22.8698, 91.0991],
    "Pabna": [24.0064, 89.2372],
    "Patuakhali": [22.3597, 90.3297],
    "Rajshahi": [24.3745, 88.6042],
    "Rangpur": [25.7439, 89.2752],
    "Satkhira": [22.7185, 89.0705],
    "Sherpur": [25.0200, 90.0153],
    "Sirajganj": [24.4583, 89.7083],
    "Sunamganj": [25.0658, 91.3950],
    "Sylhet": [24.8949, 91.8687],
    "Thakurgaon": [26.0337, 88.4617]
}

def safe_minmax(series):
    series = pd.to_numeric(series, errors="coerce")
    min_v = series.min()
    max_v = series.max()

    if pd.isna(min_v) or pd.isna(max_v) or max_v == min_v:
        return pd.Series(np.zeros(len(series)), index=series.index)

    return (series - min_v) / (max_v - min_v)

numeric_cleanup_cols = [col for col in required_columns if col != "District"]

for col in numeric_cleanup_cols:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")
    df_model[col] = df_model[col].fillna(df_model[col].median())

df_model["District"] = df_model["District"].astype(str).str.strip()

df_model["Latitude"] = df_model["District"].map(lambda x: coords.get(x, [23.6850, 90.3563])[0])
df_model["Longitude"] = df_model["District"].map(lambda x: coords.get(x, [23.6850, 90.3563])[1])

df_model["Is_Flood_Prone_District"] = df_model["District"].isin(flood_districts).astype(int)
df_model["Is_Cyclone_Prone_District"] = df_model["District"].isin(cyclone_districts).astype(int)
df_model["Is_Drought_Prone_District"] = df_model["District"].isin(drought_districts).astype(int)

df_model["Rainfall_Norm"] = safe_minmax(df_model["Annual_Rainfall_mm"])
df_model["River_Norm"] = safe_minmax(df_model["River_Water_Level_m"])
df_model["Temperature_Norm"] = safe_minmax(df_model["Avg_Temperature_C"])
df_model["AQI_Norm"] = safe_minmax(df_model["AQI"])
df_model["Carbon_Norm"] = safe_minmax(df_model["Carbon_Emission_Metric_Tons_per_Capita"])
df_model["Forest_Loss_Norm"] = 1 - safe_minmax(df_model["Forest_Cover_Percent"])

df_model["Flood_Risk_Index"] = (
    0.35 * df_model["Rainfall_Norm"] +
    0.30 * df_model["River_Norm"] +
    0.15 * df_model["Is_Flood_Prone_District"] +
    0.10 * df_model["Forest_Loss_Norm"] +
    0.05 * df_model["Temperature_Norm"] +
    0.05 * df_model["Carbon_Norm"]
) * 10

df_model["Cyclone_Risk_Index"] = (
    0.35 * df_model["Is_Cyclone_Prone_District"] +
    0.20 * df_model["Rainfall_Norm"] +
    0.15 * df_model["Temperature_Norm"] +
    0.10 * df_model["River_Norm"] +
    0.10 * df_model["Carbon_Norm"] +
    0.10 * df_model["Forest_Loss_Norm"]
) * 10

df_model["Drought_Risk_Index"] = (
    0.30 * df_model["Is_Drought_Prone_District"] +
    0.25 * df_model["Temperature_Norm"] +
    0.20 * (1 - df_model["Rainfall_Norm"]) +
    0.10 * (1 - df_model["River_Norm"]) +
    0.10 * df_model["Forest_Loss_Norm"] +
    0.05 * df_model["Carbon_Norm"]
) * 10

df_model["Agricultural_Stress_Index"] = (
    0.30 * df_model["Drought_Risk_Index"] / 10 +
    0.20 * df_model["Flood_Risk_Index"] / 10 +
    0.15 * df_model["Temperature_Norm"] +
    0.15 * df_model["Forest_Loss_Norm"] +
    0.10 * df_model["AQI_Norm"] +
    0.10 * df_model["Carbon_Norm"]
) * 10

df_model["Composite_Vulnerability_Index"] = (
    0.30 * df_model["Flood_Risk_Index"] +
    0.25 * df_model["Cyclone_Risk_Index"] +
    0.25 * df_model["Drought_Risk_Index"] +
    0.20 * df_model["Agricultural_Stress_Index"]
)

risk_cols = [
    "Flood_Risk_Index",
    "Cyclone_Risk_Index",
    "Drought_Risk_Index",
    "Agricultural_Stress_Index",
    "Composite_Vulnerability_Index"
]

print("✅ Multi-hazard indices created.")
display(df_model[["District", "Year"] + risk_cols].head())
display(df_model[risk_cols].describe())

✅ Multi-hazard indices created.


,District,Year,Flood_Risk_Index,Cyclone_Risk_Index,Drought_Risk_Index,Agricultural_Stress_Index,Composite_Vulnerability_Index
0,Satkhira,2025,5.530791,7.400889,2.933007,5.061384,5.254988
1,Tangail,2018,3.703732,2.363915,2.822698,3.467852,3.101343
2,Sunamganj,2013,6.932538,3.284697,1.817999,3.222603,3.999956
3,Sirajganj,2005,5.686967,2.961508,2.628133,3.603559,3.824212
4,Barishal,2013,6.927656,4.884669,3.572487,6.210982,5.434782


,Flood_Risk_Index,Cyclone_Risk_Index,Drought_Risk_Index,Agricultural_Stress_Index,Composite_Vulnerability_Index
count,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,4.502584,3.749274,3.906474,4.574645,4.179641
std,1.491540,1.507878,1.450325,1.047984,0.834873
min,0.707796,0.757423,0.573442,1.665003,1.764822
25%,3.410878,2.728700,2.917592,3.830441,3.580903
50%,4.509916,3.435820,3.702374,4.608142,4.185541
75%,5.570917,4.328490,4.642240,5.320824,4.786405
max,9.335061,8.821499,9.382062,7.962556,6.566019


In [6]:
# ============================================================
# 4) Train and Validate Models for All Risk Indices
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from IPython.display import display

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_features = [
    "Year",
    "Avg_Temperature_C",
    "Annual_Rainfall_mm",
    "AQI",
    "Forest_Cover_Percent",
    "River_Water_Level_m",
    "Carbon_Emission_Metric_Tons_per_Capita",
    "Latitude",
    "Longitude",
    "Is_Flood_Prone_District",
    "Is_Cyclone_Prone_District",
    "Is_Drought_Prone_District",
    "Rainfall_Norm",
    "River_Norm",
    "Temperature_Norm",
    "AQI_Norm",
    "Carbon_Norm",
    "Forest_Loss_Norm"
]

categorical_features = ["District"]

target_columns = [
    "Flood_Risk_Index",
    "Cyclone_Risk_Index",
    "Drought_Risk_Index",
    "Agricultural_Stress_Index",
    "Composite_Vulnerability_Index"
]

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=800,
        random_state=42,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=800,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        random_state=42
    )
}

all_results = []
best_models = {}

for target_col in target_columns:
    X = df_model[numeric_features + categorical_features]
    y = df_model[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", make_onehot_encoder(), categorical_features)
        ],
        sparse_threshold=0
    )

    target_models = {}
    target_results = []

    for model_name, regressor in models.items():
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("model", regressor)
        ])

        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        result = {
            "Target": target_col,
            "Model": model_name,
            "Train_R2": r2_score(y_train, train_pred),
            "Test_R2": r2_score(y_test, test_pred),
            "MAE": mean_absolute_error(y_test, test_pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, test_pred))
        }

        target_results.append(result)
        all_results.append(result)
        target_models[model_name] = model

    target_results_df = pd.DataFrame(target_results).sort_values("Test_R2", ascending=False)
    best_model_name = target_results_df.iloc[0]["Model"]
    best_models[target_col] = target_models[best_model_name]

results_df = pd.DataFrame(all_results).sort_values(
    ["Target", "Test_R2"],
    ascending=[True, False]
)

best_summary = (
    results_df.sort_values("Test_R2", ascending=False)
    .groupby("Target", as_index=False)
    .head(1)
    .sort_values("Target")
)

print("✅ Full model leaderboard:")
display(results_df)

print("✅ Best model per target:")
display(best_summary)

print("\nInterpretation:")
for _, row in best_summary.iterrows():
    print(f"- {row['Target']}: best model = {row['Model']}, Test R² = {row['Test_R2']:.4f}, MAE = {row['MAE']:.4f}")

✅ Full model leaderboard:


,Target,Model,Train_R2,Test_R2,MAE,RMSE
11,Agricultural_Stress_Index,Gradient Boosting,0.996694,0.990042,0.083451,0.104793
10,Agricultural_Stress_Index,Extra Trees,1.000000,0.972036,0.136981,0.175607
9,Agricultural_Stress_Index,Random Forest,0.994141,0.953512,0.177823,0.226419
14,Composite_Vulnerability_Index,Gradient Boosting,0.994622,0.982806,0.088883,0.110519
13,Composite_Vulnerability_Index,Extra Trees,1.000000,0.951315,0.144974,0.185970
12,Composite_Vulnerability_Index,Random Forest,0.987975,0.907101,0.207934,0.256891
5,Cyclone_Risk_Index,Gradient Boosting,0.998916,0.996605,0.067375,0.086778
4,Cyclone_Risk_Index,Extra Trees,1.000000,0.986950,0.129429,0.170145
3,Cyclone_Risk_Index,Random Forest,0.997574,0.981283,0.160800,0.203766
8,Drought_Risk_Index,Gradient Boosting,0.998581,0.996212,0.069426,0.088231


✅ Best model per target:


,Target,Model,Train_R2,Test_R2,MAE,RMSE
11,Agricultural_Stress_Index,Gradient Boosting,0.996694,0.990042,0.083451,0.104793
14,Composite_Vulnerability_Index,Gradient Boosting,0.994622,0.982806,0.088883,0.110519
5,Cyclone_Risk_Index,Gradient Boosting,0.998916,0.996605,0.067375,0.086778
8,Drought_Risk_Index,Gradient Boosting,0.998581,0.996212,0.069426,0.088231
2,Flood_Risk_Index,Gradient Boosting,0.998345,0.995968,0.076040,0.097780



Interpretation:
- Agricultural_Stress_Index: best model = Gradient Boosting, Test R² = 0.9900, MAE = 0.0835
- Composite_Vulnerability_Index: best model = Gradient Boosting, Test R² = 0.9828, MAE = 0.0889
- Cyclone_Risk_Index: best model = Gradient Boosting, Test R² = 0.9966, MAE = 0.0674
- Drought_Risk_Index: best model = Gradient Boosting, Test R² = 0.9962, MAE = 0.0694
- Flood_Risk_Index: best model = Gradient Boosting, Test R² = 0.9960, MAE = 0.0760


In [7]:
# ============================================================
# 5) What the App Predicts
# ============================================================

explanation = pd.DataFrame({
    "Prediction Target": [
        "Flood_Risk_Index",
        "Cyclone_Risk_Index",
        "Drought_Risk_Index",
        "Agricultural_Stress_Index",
        "Composite_Vulnerability_Index"
    ],
    "Meaning": [
        "0–10 flood vulnerability score based on rainfall, river level, flood-prone district status, forest loss, temperature and carbon factors.",
        "0–10 cyclone vulnerability score based on cyclone-prone/coastal exposure, rainfall, temperature, river level, forest loss and emissions.",
        "0–10 drought vulnerability score based on drought-prone exposure, high temperature, low rainfall, low river level, forest loss and emissions.",
        "0–10 agricultural climate stress score based on drought risk, flood risk, temperature, forest loss, AQI and emissions.",
        "0–10 combined vulnerability score summarising flood, cyclone, drought and agricultural stress."
    ],
    "Safe wording": [
        "Flood Risk Index Prediction",
        "Cyclone Risk Index Prediction",
        "Drought Risk Index Prediction",
        "Agricultural Climate Stress Simulation",
        "Composite Climate Vulnerability Simulation"
    ],
    "Avoid saying": [
        "Exact flood event forecast",
        "Exact cyclone landfall/count forecast",
        "Exact drought disaster forecast",
        "Exact crop yield guarantee",
        "Official disaster early-warning system"
    ]
})

display(explanation)

,Prediction Target,Meaning,Safe wording,Avoid saying
0,Flood_Risk_Index,0–10 flood vulnerability score based on rainfa...,Flood Risk Index Prediction,Exact flood event forecast
1,Cyclone_Risk_Index,0–10 cyclone vulnerability score based on cycl...,Cyclone Risk Index Prediction,Exact cyclone landfall/count forecast
2,Drought_Risk_Index,0–10 drought vulnerability score based on drou...,Drought Risk Index Prediction,Exact drought disaster forecast
3,Agricultural_Stress_Index,0–10 agricultural climate stress score based o...,Agricultural Climate Stress Simulation,Exact crop yield guarantee
4,Composite_Vulnerability_Index,0–10 combined vulnerability score summarising ...,Composite Climate Vulnerability Simulation,Official disaster early-warning system


In [8]:
# ============================================================
# 6) Quick Scenario Prediction Test in Colab
# ============================================================

sample = df_model.sample(1, random_state=7)[numeric_features + categorical_features]
print("Sample input district:", sample["District"].iloc[0])

scenario_predictions = []

for target_col, model in best_models.items():
    pred = model.predict(sample)[0]
    scenario_predictions.append({
        "Target": target_col,
        "Predicted Score": round(pred, 3)
    })

display(pd.DataFrame(scenario_predictions))

Sample input district: Comilla


,Target,Predicted Score
0,Flood_Risk_Index,4.931
1,Cyclone_Risk_Index,4.931
2,Drought_Risk_Index,4.931
3,Agricultural_Stress_Index,4.931
4,Composite_Vulnerability_Index,4.931


## Streamlit app generation

The next cell writes the complete polished Streamlit app to `/content/app.py`. After that, run the syntax check and then the Cloudflare tunnel cell.

In [9]:
%%writefile /content/app.py

import pandas as pd
import numpy as np
import streamlit as st
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ============================================================
# Page Configuration
# ============================================================

st.set_page_config(
    page_title="Bangladesh Multi-Hazard AI Simulator",
    page_icon="🇧🇩",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ============================================================
# Custom UI Styling
# ============================================================

st.markdown(
    """
    <style>
    .stApp {
        background: linear-gradient(135deg, #0f172a 0%, #111827 45%, #172554 100%);
        color: #f8fafc;
    }

    section[data-testid="stSidebar"] {
        background: rgba(15, 23, 42, 0.96);
        border-right: 1px solid rgba(148, 163, 184, 0.25);
    }

    .main-title {
        font-size: 42px;
        font-weight: 900;
        line-height: 1.1;
        text-align: center;
        margin-bottom: 8px;
        color: #f8fafc;
    }

    .sub-title {
        font-size: 17px;
        text-align: center;
        color: #cbd5e1;
        margin-bottom: 28px;
    }

    .glass-card {
        background: rgba(255, 255, 255, 0.08);
        border: 1px solid rgba(255, 255, 255, 0.14);
        border-radius: 18px;
        padding: 20px;
        box-shadow: 0 12px 35px rgba(0, 0, 0, 0.25);
        backdrop-filter: blur(10px);
        margin-bottom: 18px;
    }

    .metric-card {
        background: linear-gradient(135deg, rgba(37, 99, 235, 0.32), rgba(14, 165, 233, 0.18));
        border: 1px solid rgba(96, 165, 250, 0.30);
        border-radius: 18px;
        padding: 18px;
        min-height: 112px;
        box-shadow: 0 10px 24px rgba(0, 0, 0, 0.22);
    }

    .metric-label {
        font-size: 13px;
        color: #cbd5e1;
        margin-bottom: 8px;
        font-weight: 600;
    }

    .metric-value {
        font-size: 30px;
        color: #ffffff;
        font-weight: 900;
        line-height: 1.1;
    }

    .metric-note {
        font-size: 12px;
        color: #bfdbfe;
        margin-top: 8px;
    }

    .warning-box {
        background: rgba(250, 204, 21, 0.16);
        border-left: 5px solid #facc15;
        color: #fef9c3;
        padding: 14px 18px;
        border-radius: 14px;
        margin: 12px 0 22px 0;
    }

    .success-box {
        background: rgba(34, 197, 94, 0.14);
        border-left: 5px solid #22c55e;
        color: #dcfce7;
        padding: 14px 18px;
        border-radius: 14px;
        margin: 12px 0 22px 0;
    }

    .small-muted {
        color: #94a3b8;
        font-size: 13px;
    }

    h1, h2, h3 {
        color: #f8fafc;
    }

    div[data-testid="stMetricValue"] {
        color: #ffffff;
    }

    div[data-testid="stDataFrame"] {
        border-radius: 14px;
        overflow: hidden;
    }
    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# Constants and Configuration
# ============================================================

DATA_PATH = "/content/Bangladesh_Environmental_Climate_Change_Impact.csv"

REQUIRED_COLUMNS = [
    "District",
    "Year",
    "Avg_Temperature_C",
    "Annual_Rainfall_mm",
    "AQI",
    "Forest_Cover_Percent",
    "Flood_Impact_Score",
    "Drought_Severity",
    "River_Water_Level_m",
    "Cyclone_Count",
    "Carbon_Emission_Metric_Tons_per_Capita",
    "Agricultural_Yield_ton_per_hectare"
]

FLOOD_DISTRICTS = [
    "Sunamganj", "Sylhet", "Kurigram", "Gaibandha",
    "Jamalpur", "Bogura", "Sirajganj", "Lalmonirhat",
    "Netrokona", "Sherpur"
]

CYCLONE_DISTRICTS = [
    "Bhola", "Patuakhali", "Satkhira", "Cox's Bazar",
    "Barguna", "Chittagong", "Chattogram", "Khulna",
    "Barisal", "Feni", "Noakhali", "Bagerhat"
]

DROUGHT_DISTRICTS = [
    "Rajshahi", "Naogaon", "Chapainawabganj",
    "Dinajpur", "Rangpur", "Kushtia",
    "Thakurgaon", "Meherpur", "Chuadanga"
]

COORDS = {
    "Bagerhat": [22.6602, 89.7895],
    "Bandarban": [22.1953, 92.2184],
    "Barguna": [22.1591, 90.1189],
    "Barisal": [22.7010, 90.3535],
    "Bhola": [22.6859, 90.6440],
    "Bogura": [24.8481, 89.3730],
    "Brahmanbaria": [23.9578, 91.1111],
    "Chandpur": [23.2333, 90.6333],
    "Chapainawabganj": [24.5965, 88.2775],
    "Chattogram": [22.3569, 91.7832],
    "Chittagong": [22.3569, 91.7832],
    "Chuadanga": [23.6402, 88.8418],
    "Comilla": [23.4682, 91.1788],
    "Cox's Bazar": [21.4272, 92.0058],
    "Dhaka": [23.8103, 90.4125],
    "Dinajpur": [25.6217, 88.6354],
    "Feni": [23.0159, 91.3976],
    "Gaibandha": [25.3288, 89.5280],
    "Gazipur": [24.0023, 90.4260],
    "Gopalganj": [23.0051, 89.8266],
    "Jamalpur": [24.9375, 89.9375],
    "Jashore": [23.1664, 89.2081],
    "Jessore": [23.1664, 89.2081],
    "Khulna": [22.8456, 89.5403],
    "Kurigram": [25.8103, 89.6487],
    "Kushtia": [23.9010, 89.1204],
    "Lalmonirhat": [25.9167, 89.4500],
    "Magura": [23.4855, 89.4198],
    "Manikganj": [23.8617, 90.0003],
    "Maulvibazar": [24.4829, 91.7774],
    "Meherpur": [23.7622, 88.6318],
    "Mymensingh": [24.7471, 90.4203],
    "Naogaon": [24.7936, 88.9318],
    "Narayanganj": [23.6238, 90.5000],
    "Narsingdi": [23.9322, 90.7154],
    "Netrokona": [24.8835, 90.7310],
    "Noakhali": [22.8698, 91.0991],
    "Pabna": [24.0064, 89.2372],
    "Patuakhali": [22.3597, 90.3297],
    "Rajshahi": [24.3745, 88.6042],
    "Rangpur": [25.7439, 89.2752],
    "Satkhira": [22.7185, 89.0705],
    "Sherpur": [25.0200, 90.0153],
    "Sirajganj": [24.4583, 89.7083],
    "Sunamganj": [25.0658, 91.3950],
    "Sylhet": [24.8949, 91.8687],
    "Thakurgaon": [26.0337, 88.4617]
}

TARGETS = {
    "🌊 Flood Risk": "Flood_Risk_Index",
    "🌪️ Cyclone Risk": "Cyclone_Risk_Index",
    "🔥 Drought Risk": "Drought_Risk_Index",
    "🌾 Agricultural Stress": "Agricultural_Stress_Index",
    "🧭 Composite Vulnerability": "Composite_Vulnerability_Index"
}

TARGET_DESCRIPTIONS = {
    "Flood_Risk_Index": "Flood vulnerability based on rainfall, river level, flood-prone district status, forest loss, temperature and carbon factors.",
    "Cyclone_Risk_Index": "Cyclone vulnerability based on cyclone-prone coastal exposure, rainfall, temperature, river level, forest loss and emissions.",
    "Drought_Risk_Index": "Drought vulnerability based on drought-prone district exposure, high temperature, low rainfall, low river level, forest loss and emissions.",
    "Agricultural_Stress_Index": "Agricultural climate stress based on drought risk, flood risk, temperature, forest loss, AQI and emissions.",
    "Composite_Vulnerability_Index": "Combined multi-hazard vulnerability score summarising flood, cyclone, drought and agricultural stress."
}


# ============================================================
# Helper Functions
# ============================================================

def safe_minmax(series):
    series = pd.to_numeric(series, errors="coerce")
    min_v = series.min()
    max_v = series.max()

    if pd.isna(min_v) or pd.isna(max_v) or max_v == min_v:
        return pd.Series(np.zeros(len(series)), index=series.index)

    return (series - min_v) / (max_v - min_v)


def scale_single_value(value, reference_series):
    min_v = float(reference_series.min())
    max_v = float(reference_series.max())

    if max_v == min_v:
        return 0.0

    scaled = (float(value) - min_v) / (max_v - min_v)
    return float(np.clip(scaled, 0.0, 1.0))


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def risk_category(score):
    if score >= 7.5:
        return "Very High", "#ef4444"
    if score >= 5.5:
        return "High", "#f97316"
    if score >= 3.5:
        return "Moderate", "#facc15"
    return "Low", "#22c55e"


def metric_card(label, value, note=""):
    st.markdown(
        f"""
        <div class="metric-card">
            <div class="metric-label">{label}</div>
            <div class="metric-value">{value}</div>
            <div class="metric-note">{note}</div>
        </div>
        """,
        unsafe_allow_html=True
    )


def clean_numeric_columns(df, columns):
    for col in columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())
    return df


# ============================================================
# Data Loading and Feature Engineering
# ============================================================

@st.cache_data
def load_and_engineer_data():
    df = pd.read_csv(DATA_PATH)

    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    numeric_cols = [col for col in REQUIRED_COLUMNS if col not in ["District"]]
    df = clean_numeric_columns(df, numeric_cols)

    df["District"] = df["District"].astype(str).str.strip()

    df["Latitude"] = df["District"].map(lambda x: COORDS.get(x, [23.6850, 90.3563])[0])
    df["Longitude"] = df["District"].map(lambda x: COORDS.get(x, [23.6850, 90.3563])[1])

    df["Is_Flood_Prone_District"] = df["District"].isin(FLOOD_DISTRICTS).astype(int)
    df["Is_Cyclone_Prone_District"] = df["District"].isin(CYCLONE_DISTRICTS).astype(int)
    df["Is_Drought_Prone_District"] = df["District"].isin(DROUGHT_DISTRICTS).astype(int)

    df["Rainfall_Norm"] = safe_minmax(df["Annual_Rainfall_mm"])
    df["River_Norm"] = safe_minmax(df["River_Water_Level_m"])
    df["Temperature_Norm"] = safe_minmax(df["Avg_Temperature_C"])
    df["AQI_Norm"] = safe_minmax(df["AQI"])
    df["Carbon_Norm"] = safe_minmax(df["Carbon_Emission_Metric_Tons_per_Capita"])
    df["Forest_Loss_Norm"] = 1 - safe_minmax(df["Forest_Cover_Percent"])

    df["Flood_Risk_Index"] = (
        0.35 * df["Rainfall_Norm"] +
        0.30 * df["River_Norm"] +
        0.15 * df["Is_Flood_Prone_District"] +
        0.10 * df["Forest_Loss_Norm"] +
        0.05 * df["Temperature_Norm"] +
        0.05 * df["Carbon_Norm"]
    ) * 10

    df["Cyclone_Risk_Index"] = (
        0.35 * df["Is_Cyclone_Prone_District"] +
        0.20 * df["Rainfall_Norm"] +
        0.15 * df["Temperature_Norm"] +
        0.10 * df["River_Norm"] +
        0.10 * df["Carbon_Norm"] +
        0.10 * df["Forest_Loss_Norm"]
    ) * 10

    df["Drought_Risk_Index"] = (
        0.30 * df["Is_Drought_Prone_District"] +
        0.25 * df["Temperature_Norm"] +
        0.20 * (1 - df["Rainfall_Norm"]) +
        0.10 * (1 - df["River_Norm"]) +
        0.10 * df["Forest_Loss_Norm"] +
        0.05 * df["Carbon_Norm"]
    ) * 10

    df["Agricultural_Stress_Index"] = (
        0.30 * df["Drought_Risk_Index"] / 10 +
        0.20 * df["Flood_Risk_Index"] / 10 +
        0.15 * df["Temperature_Norm"] +
        0.15 * df["Forest_Loss_Norm"] +
        0.10 * df["AQI_Norm"] +
        0.10 * df["Carbon_Norm"]
    ) * 10

    df["Composite_Vulnerability_Index"] = (
        0.30 * df["Flood_Risk_Index"] +
        0.25 * df["Cyclone_Risk_Index"] +
        0.25 * df["Drought_Risk_Index"] +
        0.20 * df["Agricultural_Stress_Index"]
    )

    return df


NUMERIC_FEATURES = [
    "Year",
    "Avg_Temperature_C",
    "Annual_Rainfall_mm",
    "AQI",
    "Forest_Cover_Percent",
    "River_Water_Level_m",
    "Carbon_Emission_Metric_Tons_per_Capita",
    "Latitude",
    "Longitude",
    "Is_Flood_Prone_District",
    "Is_Cyclone_Prone_District",
    "Is_Drought_Prone_District",
    "Rainfall_Norm",
    "River_Norm",
    "Temperature_Norm",
    "AQI_Norm",
    "Carbon_Norm",
    "Forest_Loss_Norm"
]

CATEGORICAL_FEATURES = ["District"]


@st.cache_resource
def train_all_models(df):
    models = {
        "Random Forest": RandomForestRegressor(
            n_estimators=800,
            random_state=42,
            n_jobs=-1
        ),
        "Extra Trees": ExtraTreesRegressor(
            n_estimators=800,
            random_state=42,
            n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=3,
            random_state=42
        )
    }

    results = []
    best_models = {}

    X_all = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]

    for target_col in TARGETS.values():
        y_all = df[target_col]

        X_train, X_test, y_train, y_test = train_test_split(
            X_all,
            y_all,
            test_size=0.20,
            random_state=42
        )

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), NUMERIC_FEATURES),
                ("cat", make_onehot_encoder(), CATEGORICAL_FEATURES)
            ],
            sparse_threshold=0
        )

        target_models = {}

        for model_name, regressor in models.items():
            model = Pipeline([
                ("preprocessor", preprocessor),
                ("model", regressor)
            ])

            model.fit(X_train, y_train)

            train_pred = model.predict(X_train)
            test_pred = model.predict(X_test)

            row = {
                "Target": target_col,
                "Model": model_name,
                "Train_R2": r2_score(y_train, train_pred),
                "Test_R2": r2_score(y_test, test_pred),
                "MAE": mean_absolute_error(y_test, test_pred),
                "RMSE": np.sqrt(mean_squared_error(y_test, test_pred))
            }

            results.append(row)
            target_models[model_name] = model

        target_results = pd.DataFrame([r for r in results if r["Target"] == target_col])
        best_model_name = target_results.sort_values("Test_R2", ascending=False).iloc[0]["Model"]
        best_models[target_col] = target_models[best_model_name]

    results_df = pd.DataFrame(results).sort_values(["Target", "Test_R2"], ascending=[True, False])
    best_summary = (
        results_df.sort_values("Test_R2", ascending=False)
        .groupby("Target", as_index=False)
        .head(1)
        .sort_values("Target")
    )

    return best_models, results_df, best_summary


def build_scenario_row(df, district, year, temperature, rainfall, aqi, forest, river, carbon):
    lat = COORDS.get(district, [23.6850, 90.3563])[0]
    lon = COORDS.get(district, [23.6850, 90.3563])[1]

    is_flood = int(district in FLOOD_DISTRICTS)
    is_cyclone = int(district in CYCLONE_DISTRICTS)
    is_drought = int(district in DROUGHT_DISTRICTS)

    rainfall_norm = scale_single_value(rainfall, df["Annual_Rainfall_mm"])
    river_norm = scale_single_value(river, df["River_Water_Level_m"])
    temperature_norm = scale_single_value(temperature, df["Avg_Temperature_C"])
    aqi_norm = scale_single_value(aqi, df["AQI"])
    carbon_norm = scale_single_value(carbon, df["Carbon_Emission_Metric_Tons_per_Capita"])
    forest_loss_norm = 1 - scale_single_value(forest, df["Forest_Cover_Percent"])

    scenario = pd.DataFrame([{
        "Year": year,
        "Avg_Temperature_C": temperature,
        "Annual_Rainfall_mm": rainfall,
        "AQI": aqi,
        "Forest_Cover_Percent": forest,
        "River_Water_Level_m": river,
        "Carbon_Emission_Metric_Tons_per_Capita": carbon,
        "Latitude": lat,
        "Longitude": lon,
        "Is_Flood_Prone_District": is_flood,
        "Is_Cyclone_Prone_District": is_cyclone,
        "Is_Drought_Prone_District": is_drought,
        "Rainfall_Norm": rainfall_norm,
        "River_Norm": river_norm,
        "Temperature_Norm": temperature_norm,
        "AQI_Norm": aqi_norm,
        "Carbon_Norm": carbon_norm,
        "Forest_Loss_Norm": forest_loss_norm,
        "District": district
    }])

    direct_indices = {
        "Flood_Risk_Index": (
            0.35 * rainfall_norm +
            0.30 * river_norm +
            0.15 * is_flood +
            0.10 * forest_loss_norm +
            0.05 * temperature_norm +
            0.05 * carbon_norm
        ) * 10,
        "Cyclone_Risk_Index": (
            0.35 * is_cyclone +
            0.20 * rainfall_norm +
            0.15 * temperature_norm +
            0.10 * river_norm +
            0.10 * carbon_norm +
            0.10 * forest_loss_norm
        ) * 10,
        "Drought_Risk_Index": (
            0.30 * is_drought +
            0.25 * temperature_norm +
            0.20 * (1 - rainfall_norm) +
            0.10 * (1 - river_norm) +
            0.10 * forest_loss_norm +
            0.05 * carbon_norm
        ) * 10
    }

    direct_indices["Agricultural_Stress_Index"] = (
        0.30 * direct_indices["Drought_Risk_Index"] / 10 +
        0.20 * direct_indices["Flood_Risk_Index"] / 10 +
        0.15 * temperature_norm +
        0.15 * forest_loss_norm +
        0.10 * aqi_norm +
        0.10 * carbon_norm
    ) * 10

    direct_indices["Composite_Vulnerability_Index"] = (
        0.30 * direct_indices["Flood_Risk_Index"] +
        0.25 * direct_indices["Cyclone_Risk_Index"] +
        0.25 * direct_indices["Drought_Risk_Index"] +
        0.20 * direct_indices["Agricultural_Stress_Index"]
    )

    return scenario, direct_indices


# ============================================================
# Load Data
# ============================================================

try:
    df = load_and_engineer_data()
except FileNotFoundError:
    st.error("CSV file not found. Please upload Bangladesh_Environmental_Climate_Change_Impact.csv into /content/.")
    st.stop()
except ValueError as err:
    st.error(str(err))
    st.stop()

best_models, results_df, best_summary = train_all_models(df)
latest_year = int(df["Year"].max())
latest_df = df[df["Year"] == latest_year].copy()


# ============================================================
# Header
# ============================================================

st.markdown(
    '<div class="main-title">🇧🇩 Bangladesh Multi-Hazard Risk & Climate Vulnerability Simulator</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="sub-title">Dynamic AI-assisted risk-index simulator for flood, cyclone, drought and agricultural climate stress.</div>',
    unsafe_allow_html=True
)

st.markdown(
    """
    <div class="warning-box">
        <b>Important:</b> This app predicts transparent 0–10 risk indices. It does not predict exact disaster events,
        flood depth, cyclone landfall, casualty numbers or government-grade forecasts.
    </div>
    """,
    unsafe_allow_html=True
)


# ============================================================
# Sidebar
# ============================================================

st.sidebar.markdown("## ⚙️ Control Panel")
page = st.sidebar.radio(
    "Navigation Menu",
    [
        "🏠 Executive Dashboard",
        "🗺️ Multi-Hazard Risk Radar",
        "🔮 AI Hazard Predictor",
        "🌾 Agricultural Impact Simulator",
        "📍 District Explorer",
        "📊 Dataset Workspace"
    ]
)

st.sidebar.markdown("---")
selected_year = st.sidebar.slider(
    "Dataset Year",
    int(df["Year"].min()),
    int(df["Year"].max()),
    latest_year
)

districts = sorted(df["District"].unique())
selected_district = st.sidebar.selectbox("District Focus", districts)

year_df = df[df["Year"] == selected_year].copy()


# ============================================================
# Page 1: Executive Dashboard
# ============================================================

if page == "🏠 Executive Dashboard":
    st.header("🏠 Executive Dashboard")

    avg_flood = year_df["Flood_Risk_Index"].mean()
    avg_cyclone = year_df["Cyclone_Risk_Index"].mean()
    avg_drought = year_df["Drought_Risk_Index"].mean()
    avg_agri = year_df["Agricultural_Stress_Index"].mean()

    c1, c2, c3, c4 = st.columns(4)

    with c1:
        metric_card("Average Flood Risk", f"{avg_flood:.2f}/10", f"Selected year: {selected_year}")
    with c2:
        metric_card("Average Cyclone Risk", f"{avg_cyclone:.2f}/10", f"Selected year: {selected_year}")
    with c3:
        metric_card("Average Drought Risk", f"{avg_drought:.2f}/10", f"Selected year: {selected_year}")
    with c4:
        metric_card("Agricultural Stress", f"{avg_agri:.2f}/10", f"Selected year: {selected_year}")

    st.markdown("### 🧭 Top Composite Vulnerability Districts")

    top_composite = (
        year_df.groupby("District", as_index=False)
        .agg({
            "Composite_Vulnerability_Index": "mean",
            "Flood_Risk_Index": "mean",
            "Cyclone_Risk_Index": "mean",
            "Drought_Risk_Index": "mean",
            "Agricultural_Stress_Index": "mean",
            "Latitude": "mean",
            "Longitude": "mean"
        })
        .sort_values("Composite_Vulnerability_Index", ascending=False)
        .head(12)
    )

    fig = px.bar(
        top_composite,
        x="Composite_Vulnerability_Index",
        y="District",
        orientation="h",
        title="Top 12 Districts by Composite Vulnerability",
        text="Composite_Vulnerability_Index"
    )
    fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
    fig.update_layout(
        height=500,
        yaxis={"categoryorder": "total ascending"},
        margin=dict(l=20, r=30, t=60, b=20)
    )
    st.plotly_chart(fig, use_container_width=True)

    st.markdown("### 🗺️ Risk Map")

    map_metric_label = st.selectbox(
        "Choose map layer",
        list(TARGETS.keys()),
        index=4
    )
    map_metric = TARGETS[map_metric_label]

    district_map = (
        year_df.groupby("District", as_index=False)
        .agg({
            map_metric: "mean",
            "Latitude": "mean",
            "Longitude": "mean"
        })
    )

    fig_map = px.scatter_mapbox(
        district_map,
        lat="Latitude",
        lon="Longitude",
        color=map_metric,
        size=map_metric,
        hover_name="District",
        zoom=5.8,
        height=560,
        mapbox_style="open-street-map",
        title=f"{map_metric_label} Distribution"
    )
    fig_map.update_layout(margin=dict(l=0, r=0, t=45, b=0))
    st.plotly_chart(fig_map, use_container_width=True)


# ============================================================
# Page 2: Multi-Hazard Risk Radar
# ============================================================

elif page == "🗺️ Multi-Hazard Risk Radar":
    st.header("🗺️ Multi-Hazard Risk Radar")

    hazard_label = st.selectbox("Select hazard/risk layer", list(TARGETS.keys()))
    hazard_col = TARGETS[hazard_label]

    st.markdown(
        f"""
        <div class="glass-card">
            <b>{hazard_label}</b><br>
            <span class="small-muted">{TARGET_DESCRIPTIONS[hazard_col]}</span>
        </div>
        """,
        unsafe_allow_html=True
    )

    district_risk = (
        year_df.groupby("District", as_index=False)
        .agg({
            hazard_col: "mean",
            "Latitude": "mean",
            "Longitude": "mean"
        })
        .sort_values(hazard_col, ascending=False)
    )

    left, right = st.columns([1.2, 1])

    with left:
        top_n = st.slider("Number of districts to display", 5, 25, 12)

        fig = px.bar(
            district_risk.head(top_n),
            x=hazard_col,
            y="District",
            orientation="h",
            title=f"Top {top_n} Districts — {hazard_label}",
            text=hazard_col
        )
        fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
        fig.update_layout(
            height=530,
            yaxis={"categoryorder": "total ascending"},
            margin=dict(l=20, r=30, t=60, b=20)
        )
        st.plotly_chart(fig, use_container_width=True)

    with right:
        st.markdown("#### District Ranking Table")
        display_cols = ["District", hazard_col]
        st.dataframe(
            district_risk[display_cols].head(top_n),
            use_container_width=True,
            hide_index=True
        )

    st.markdown("### 🔗 Cross-Hazard Correlation Matrix")

    corr_cols = [
        "Flood_Risk_Index",
        "Cyclone_Risk_Index",
        "Drought_Risk_Index",
        "Agricultural_Stress_Index",
        "Composite_Vulnerability_Index",
        "Annual_Rainfall_mm",
        "River_Water_Level_m",
        "Avg_Temperature_C",
        "Forest_Cover_Percent",
        "AQI"
    ]

    corr = df[corr_cols].corr(numeric_only=True)

    fig_corr = px.imshow(
        corr,
        text_auto=".2f",
        aspect="auto",
        title="Correlation Between Risk Indices and Environmental Indicators"
    )
    fig_corr.update_layout(height=620)
    st.plotly_chart(fig_corr, use_container_width=True)


# ============================================================
# Page 3: AI Hazard Predictor
# ============================================================

elif page == "🔮 AI Hazard Predictor":
    st.header("🔮 AI Hazard Predictor")

    st.markdown(
        """
        <div class="success-box">
            This page predicts 0–10 risk-index scores for a custom climate scenario.
            The model uses district, year, temperature, rainfall, AQI, forest cover, river level and carbon emissions.
        </div>
        """,
        unsafe_allow_html=True
    )

    target_label = st.selectbox("Choose target to predict", list(TARGETS.keys()))
    target_col = TARGETS[target_label]

    model_row = best_summary[best_summary["Target"] == target_col].iloc[0]

    c1, c2, c3 = st.columns(3)
    with c1:
        metric_card("Best Model", str(model_row["Model"]), "Selected automatically by Test R²")
    with c2:
        metric_card("Test R² Score", f"{model_row['Test_R2']:.4f}", "Real test score, not hardcoded")
    with c3:
        metric_card("MAE", f"{model_row['MAE']:.4f}", "Average prediction error")

    st.markdown("### Scenario Inputs")

    s1, s2, s3 = st.columns(3)

    with s1:
        p_district = st.selectbox("District", districts, index=districts.index(selected_district))
        p_year = st.slider("Year", int(df["Year"].min()), int(df["Year"].max()), selected_year)
        p_temperature = st.slider("Average Temperature (°C)", 15.0, 45.0, 28.0, 0.1)

    with s2:
        p_rainfall = st.slider("Annual Rainfall (mm)", 500.0, 5000.0, 2200.0, 10.0)
        p_river = st.slider("River Water Level (m)", 0.0, 20.0, 5.0, 0.1)
        p_forest = st.slider("Forest Cover (%)", 1.0, 50.0, 15.0, 0.1)

    with s3:
        p_aqi = st.slider("Air Quality Index", 10.0, 300.0, 110.0, 1.0)
        p_carbon = st.slider("Carbon Emission Metric Tons per Capita", 0.1, 5.0, 1.2, 0.1)

    scenario_row, direct_indices = build_scenario_row(
        df,
        p_district,
        p_year,
        p_temperature,
        p_rainfall,
        p_aqi,
        p_forest,
        p_river,
        p_carbon
    )

    if st.button("🚀 Run Multi-Hazard AI Simulation", use_container_width=True):
        selected_prediction = float(best_models[target_col].predict(scenario_row)[0])
        category, cat_color = risk_category(selected_prediction)

        st.markdown("### Prediction Result")

        r1, r2 = st.columns(2)
        with r1:
            metric_card(target_label, f"{selected_prediction:.2f}/10", "AI model prediction")
        with r2:
            metric_card("Risk Category", category, "Low / Moderate / High / Very High")

        st.markdown("### All-Hazard Scenario Output")

        scenario_predictions = []
        for label, col in TARGETS.items():
            pred = float(best_models[col].predict(scenario_row)[0])
            cat, _ = risk_category(pred)
            scenario_predictions.append({
                "Risk Target": label,
                "Predicted Score": round(pred, 3),
                "Direct Index Logic": round(direct_indices[col], 3),
                "Risk Category": cat
            })

        scenario_df = pd.DataFrame(scenario_predictions)
        st.dataframe(scenario_df, use_container_width=True, hide_index=True)

        fig_scenario = px.bar(
            scenario_df,
            x="Risk Target",
            y="Predicted Score",
            title=f"Multi-Hazard Scenario Profile — {p_district}",
            text="Predicted Score"
        )
        fig_scenario.update_traces(texttemplate="%{text:.2f}", textposition="outside")
        fig_scenario.update_layout(yaxis_range=[0, 10], height=460)
        st.plotly_chart(fig_scenario, use_container_width=True)


# ============================================================
# Page 4: Agricultural Impact Simulator
# ============================================================

elif page == "🌾 Agricultural Impact Simulator":
    st.header("🌾 Agricultural Impact Simulator")

    st.markdown(
        """
        <div class="glass-card">
            Agricultural stress is estimated from drought risk, flood risk, temperature, forest loss, AQI and carbon emissions.
            A higher score means stronger climate pressure on agricultural resilience.
        </div>
        """,
        unsafe_allow_html=True
    )

    agri_year = year_df.copy()

    district_agri = (
        agri_year.groupby("District", as_index=False)
        .agg({
            "Agricultural_Stress_Index": "mean",
            "Agricultural_Yield_ton_per_hectare": "mean",
            "Drought_Risk_Index": "mean",
            "Flood_Risk_Index": "mean",
            "Latitude": "mean",
            "Longitude": "mean"
        })
        .sort_values("Agricultural_Stress_Index", ascending=False)
    )

    col1, col2 = st.columns([1.1, 1])

    with col1:
        fig = px.scatter(
            district_agri,
            x="Agricultural_Stress_Index",
            y="Agricultural_Yield_ton_per_hectare",
            size="Drought_Risk_Index",
            hover_name="District",
            title="Agricultural Stress vs Yield",
            labels={
                "Agricultural_Stress_Index": "Agricultural Stress Index",
                "Agricultural_Yield_ton_per_hectare": "Yield tons/hectare"
            }
        )
        fig.update_layout(height=520)
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        st.markdown("#### Highest Agricultural Stress")
        st.dataframe(
            district_agri[[
                "District",
                "Agricultural_Stress_Index",
                "Agricultural_Yield_ton_per_hectare",
                "Drought_Risk_Index",
                "Flood_Risk_Index"
            ]].head(15),
            use_container_width=True,
            hide_index=True
        )


# ============================================================
# Page 5: District Explorer
# ============================================================

elif page == "📍 District Explorer":
    st.header("📍 District Explorer")

    district_data = df[df["District"] == selected_district].sort_values("Year").copy()

    if district_data.empty:
        st.warning("No data found for selected district.")
        st.stop()

    latest_district = district_data[district_data["Year"] == district_data["Year"].max()].iloc[0]

    c1, c2, c3, c4 = st.columns(4)
    with c1:
        metric_card("Flood Risk", f"{latest_district['Flood_Risk_Index']:.2f}/10", selected_district)
    with c2:
        metric_card("Cyclone Risk", f"{latest_district['Cyclone_Risk_Index']:.2f}/10", selected_district)
    with c3:
        metric_card("Drought Risk", f"{latest_district['Drought_Risk_Index']:.2f}/10", selected_district)
    with c4:
        metric_card("Composite Vulnerability", f"{latest_district['Composite_Vulnerability_Index']:.2f}/10", selected_district)

    st.markdown("### Historical Risk Trends")

    trend_cols = [
        "Flood_Risk_Index",
        "Cyclone_Risk_Index",
        "Drought_Risk_Index",
        "Agricultural_Stress_Index",
        "Composite_Vulnerability_Index"
    ]

    trend_df = district_data[["Year"] + trend_cols].melt(
        id_vars="Year",
        value_vars=trend_cols,
        var_name="Risk Type",
        value_name="Score"
    )

    fig_trend = px.line(
        trend_df,
        x="Year",
        y="Score",
        color="Risk Type",
        markers=True,
        title=f"Risk Index Trends — {selected_district}"
    )
    fig_trend.update_layout(height=520, yaxis_range=[0, 10])
    st.plotly_chart(fig_trend, use_container_width=True)

    st.markdown("### Environmental Indicator Trends")

    env_cols = [
        "Avg_Temperature_C",
        "Annual_Rainfall_mm",
        "River_Water_Level_m",
        "AQI",
        "Forest_Cover_Percent"
    ]

    selected_env = st.multiselect(
        "Choose environmental indicators",
        env_cols,
        default=["Avg_Temperature_C", "Annual_Rainfall_mm", "River_Water_Level_m"]
    )

    if selected_env:
        env_df = district_data[["Year"] + selected_env].melt(
            id_vars="Year",
            value_vars=selected_env,
            var_name="Indicator",
            value_name="Value"
        )

        fig_env = px.line(
            env_df,
            x="Year",
            y="Value",
            color="Indicator",
            markers=True,
            title=f"Environmental Indicator Trends — {selected_district}"
        )
        fig_env.update_layout(height=520)
        st.plotly_chart(fig_env, use_container_width=True)


# ============================================================
# Page 6: Dataset Workspace
# ============================================================

elif page == "📊 Dataset Workspace":
    st.header("📊 Dataset Workspace")

    st.markdown("### Model Leaderboard")
    st.dataframe(results_df, use_container_width=True, hide_index=True)

    st.markdown("### Best Model per Target")
    st.dataframe(best_summary, use_container_width=True, hide_index=True)

    st.markdown("### Dataset Search and Download")

    query = st.text_input("Search district records")
    if query:
        filtered_df = df[df["District"].str.contains(query, case=False, na=False)].copy()
    else:
        filtered_df = df.copy()

    selected_cols = st.multiselect(
        "Choose columns to display",
        list(filtered_df.columns),
        default=[
            "District",
            "Year",
            "Flood_Risk_Index",
            "Cyclone_Risk_Index",
            "Drought_Risk_Index",
            "Agricultural_Stress_Index",
            "Composite_Vulnerability_Index"
        ]
    )

    if selected_cols:
        st.dataframe(filtered_df[selected_cols], use_container_width=True)
    else:
        st.dataframe(filtered_df, use_container_width=True)

    csv_data = filtered_df.to_csv(index=False).encode("utf-8")

    st.download_button(
        label="📥 Download Filtered Dataset",
        data=csv_data,
        file_name="bangladesh_multi_hazard_risk_dataset.csv",
        mime="text/csv",
        use_container_width=True
    )

    st.markdown(
        """
        <div class="warning-box">
            Raw fields such as Flood_Impact_Score, Cyclone_Count and Drought_Severity are kept in the dataset,
            but the prediction pages use transparent engineered risk indices because the raw targets may be noisy.
        </div>
        """,
        unsafe_allow_html=True
    )


Writing /content/app.py


^C
✅ app.py patched for faster Streamlit loading


ok

<IPython.core.display.Javascript object>

## Final presentation wording

Use: **Bangladesh Multi-Hazard Risk & Climate Vulnerability Simulator**.

Say it predicts: **0–10 risk-index scores for flood, cyclone, drought and agricultural climate stress**.

Avoid saying it predicts exact real-world disaster events.